In [1]:
from gam_rs_utils.utils import *
from src.rset_opt import RSetOPT
from src.run_app import get_models_from_rset
from method_scripts.results import Results

In [2]:
dn = 'bank'
data = pd.read_csv(f'datasets/{dn}.csv')
l0 = 0.001
l2 = 0.001
m = 1.01

# data is 17 features
# w becomes 3719 features

# feature: house
# 3  7 12
# 0  0  1 0 0
# 0  0  0 0 1

# 3,4,5,6,7
# 1 1 1 0 0
# 1 1 1 1 1

fastsparse_data = Results.create_fastsparse_dataset(dn, l0, l2)
bin_X = fastsparse_data['bin_X']
y = fastsparse_data['y']
cum_header = fastsparse_data['cum_header']
w = fastsparse_data['w']

print("data shape", data.shape)
print("w", len(w), "y", len(y), "header", len(cum_header))
print("bin_X shape", bin_X.shape)

loading dataset from results/bank/l0_0.001_l2_0.001.pkl
Loading cached fastsparse dataset for bank with l0=0.001 and l2=0.001
data shape (4521, 17)
w 3719 y 4521 header 3719
bin_X shape (4521, 3719)


In [4]:
w.nonzero()[0], w

(array([   0, 2442, 2444, 2447, 2489, 2699, 2836, 3093, 3182, 3660, 3691,
        3692, 3716]),
 array([ 4.67773085,  0.        ,  0.        , ..., -2.00090441,
         0.        ,  0.        ]))

In [3]:
sparse_X, sparse_header = utils.binary_to_one_hot(data.iloc[:,:-1], w, cum_header)
sparse_header, len(sparse_header), w.nonzero()[0].shape

(['intercept',
  'housing<=0.0',
  '0.0<housing<=1',
  'loan<=0.0',
  '0.0<loan<=1',
  'contact<=1.0',
  '1.0<contact<=2',
  'month<=9.0',
  '9.0<month<=11',
  'duration<=211.0',
  '211.0<duration<=348.0',
  '348.0<duration<=645.0',
  '645.0<duration<=770.0',
  '770.0<duration<=3025',
  'pdays<=374.0',
  '374.0<pdays<=871',
  'previous<=0.0',
  '0.0<previous<=1.0',
  '1.0<previous<=25',
  'poutcome<=1.0',
  '1.0<poutcome<=3'],
 21,
 (13,))

In [7]:
# start = time()

sparse_X, sparse_header = utils.binary_to_one_hot(data.iloc[:,:-1], w, cum_header)
sparse_gam_file = prepare_sparse_gam(dn, l0, l2, m, sparse_X, y, cum_header, sparse_header)

model = RSetOPT(sparse_gam_file)
model.finetune_ellipsoid()
H_opt = model.get_normalized_H()
model.update_file(H_opt, model.w_orig)

# end = time()

with open(sparse_gam_file, 'rb') as f:
    sparse_gam_data = pickle.load(f)

objective: 0.2443484497913396 objective in LR 0.2443484497913396
m:1.01, log objective:0.2443484497913396, eps:0.246791934289253
----------- before optimization -----------
volume proportional to  tensor(3.5345e+24, dtype=torch.float64, grad_fn=<MulBackward0>)
----------- after optimization -----------
volume proportional to  tensor(1.4667e+24, dtype=torch.float64, grad_fn=<MulBackward0>)


In [18]:
sparse_gam_data['w_orig'].shape

(21,)

In [8]:
n_samples = 100
sampling = 'uniform'
distance_metric = None
r_min = None

w_samples, rset = get_models_from_rset(
    sparse_gam_file, n_samples=n_samples, plot_shape=False, 
    sampling=sampling, distance_metric=distance_metric, r_min=r_min,
)

print('w_samples shape', w_samples.shape)

w_samples shape (100, 21)


In [5]:
# w_samples_zeroed = ModelUtils.hard_threshold_samples(w_samples, rset, 17)
# ModelUtils.print_results_summary(w_samples_zeroed, sparse_gam_data['w_opt'], X, y, l2, sample_p, end - start)

array([-2.04192107, -2.44651175, -3.86067603, ..., -3.22057456,
       -2.72400194, -2.44651175])

In [9]:
header_object = ModelUtils.get_header_object(cum_header)
sparse_header_object = ModelUtils.get_header_object(sparse_header)

expanded_w_samples = ModelUtils.expand_w_samples(w_samples, sparse_header_object, header_object)

In [10]:
for i in range(len(w_samples)):
    expanded_w = expanded_w_samples[i]
    w = w_samples[i]
    a = ModelUtils.get_logits(sparse_X, w)
    b = ModelUtils.get_logits(bin_X, expanded_w)
    if not np.allclose(a,b):
        print(f'w_samples[{i}] is not equal to expanded_w')
        print(a - b)

In [11]:
loss_a, _ = ModelUtils.get_loss(
    bin_X, y, 
    expanded_w_samples, 
    loss_type='logistic', l2=0.001, 
)
loss_b, _ = ModelUtils.get_loss(
    sparse_X, y, 
    w_samples, 
    loss_type='logistic', l2=0.001, 
)
np.allclose(loss_a, loss_b)

True